# Phase 2 baseline demo — tabular brain-age model

Walks the full Pillar 2 flow end to end: DB → Parquet export → region-wise
feature matrix (TIV/sex-adjusted) → subject-grouped CV harness → pluggable
bias correction → MLflow-logged leaderboard.

See `docs/DESIGN.md` §4.1 for the harness design and §3.2 for the schema.
No subject IDs or raw feature values are printed below — only aggregate
counts and metrics. Outputs are stripped on commit (nbstripout, see
`.pre-commit-config.yaml`) regardless.

In [ ]:
import pandas as pd

from bagpipe.core.config import REPO_ROOT, get_path
from bagpipe.db.export_training_table import export as export_training_table
from bagpipe.models.tabular import build_region_matrix
from bagpipe.models.evaluate import evaluate
from bagpipe.models.bias_correction import get_corrector
from bagpipe.models.covariate_adjustment import TIVSexAdjustedRegressor
from bagpipe.models import baseline

from sklearn.linear_model import LinearRegression, RidgeCV
import numpy as np

## 1. Export the analytical Parquet tables

Materializes `globals.parquet` / `regional.parquet` / `image_paths.parquet`
from the `features` long-format store (`bag export training-table`, same as
the CLI). Re-run any time the DB changes — it's a full regenerate, not an
incremental cache.

In [ ]:
export_summary = export_training_table()
pd.DataFrame(export_summary).T[["rows"]]

## 2. Build the region-wise feature matrix

One column per (atlas, region) **GM volume only** (`metrics=["vol_gm"]`),
plus `TIV` and `sex` as the last two columns — the layout
`TIVSexAdjustedRegressor` expects. GM-only is deliberate: these are flat
single-metric models, so mixing GM/WM/CSF into one undifferentiated vector
per region isn't meaningful here — that's what the stacked ensemble
(`stacked_ensemble_review.ipynb`) is for, since it groups all three metrics
per region before flattening. Target is chronological age; CV groups are
`subject_key`, so a subject's repeated sessions never span train/test.

In [ ]:
X, y, groups, region_columns = build_region_matrix(get_path("datasets_dir"), metrics=["vol_gm"])
print(f"samples: {X.shape[0]}, region features: {len(region_columns)}, unique subjects: {len(set(groups))}")

## 3. Run the eval harness

Two candidates through the identical grouped-CV split: a plain linear model
and Ridge (alpha tuned internally via `RidgeCV`), both wrapped in
`TIVSexAdjustedRegressor` (regions residualized against TIV + sex, fit on
the training fold only) and both bias-corrected with the de Lange/Cole
method. Non-negotiable rules from `DESIGN.md` §4.1: subject-grouped splits,
MAE primary metric, raw + corrected BAG both kept.

In [ ]:
candidates = {
    "linear": lambda: LinearRegression(),
    "ridge": lambda: RidgeCV(alphas=np.logspace(-3, 3, 13)),
}

results = {}
for name, base_model_fn in candidates.items():
    model_fn = lambda base_model_fn=base_model_fn: TIVSexAdjustedRegressor(base_model_fn)
    results[name] = evaluate(
        model_fn, X, y, groups, n_splits=5, bias_corrector=get_corrector("cole")
    )

## 4. Leaderboard

Same splits, same bias-correction method — metrics are directly comparable.

In [ ]:
leaderboard = pd.DataFrame({name: r.metrics for name, r in results.items()}).T
leaderboard.sort_values("mae_corrected")

## 4b. Diagnostics — predictions, BAG, distributions, sex differences

Best model from the leaderboard above (Ridge). Only aggregate plots and
group-level statistics below — no per-subject identifiers.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

best_model = leaderboard.sort_values("mae_corrected").index[0]
pred = results[best_model].predictions.copy()
pred["sex"] = X[:, -1][pred["index"].to_numpy()]
pred["sex_label"] = pred["sex"].map({0.0: "Male", 1.0: "Female"})
pred["bag_raw"] = pred["y_pred_raw"] - pred["y_true"]
pred["bag_corrected"] = pred["y_pred_corrected"] - pred["y_true"]
print(f"{len(pred)} test predictions, {best_model} model")

### Predicted vs. true age

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, col, title in zip(
    axes, ["y_pred_raw", "y_pred_corrected"], ["Raw predictions", "Cole-corrected predictions"]
):
    ax.scatter(pred["y_true"], pred[col], s=8, alpha=0.3)
    lims = [pred["y_true"].min(), pred["y_true"].max()]
    ax.plot(lims, lims, "k--", lw=1, label="y = x")
    ax.set_xlabel("True age")
    ax.set_ylabel("Predicted age")
    ax.set_title(f"{title} ({best_model})")
    ax.legend()
plt.tight_layout()
plt.show()

### BAG vs. true age — the age-bias trend correction should remove

Regression-to-the-mean shows up as a nonzero slope of BAG against true age;
Cole correction is fit exactly to zero that slope out.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, col, title in zip(axes, ["bag_raw", "bag_corrected"], ["Raw BAG", "Corrected BAG"]):
    ax.scatter(pred["y_true"], pred[col], s=8, alpha=0.3)
    slope, intercept, r, p, se = stats.linregress(pred["y_true"], pred[col])
    xs = np.array([pred["y_true"].min(), pred["y_true"].max()])
    ax.plot(xs, slope * xs + intercept, "r-", lw=2, label=f"slope={slope:.3f}, p={p:.1e}")
    ax.axhline(0, color="k", lw=1, ls=":")
    ax.set_xlabel("True age")
    ax.set_ylabel("BAG (predicted - true)")
    ax.set_title(f"{title} ({best_model})")
    ax.legend()
plt.tight_layout()
plt.show()

### BAG distribution — raw vs. corrected

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.kdeplot(pred["bag_raw"], label="raw", ax=ax)
sns.kdeplot(pred["bag_corrected"], label="corrected", ax=ax)
ax.axvline(0, color="k", lw=1, ls=":")
ax.set_xlabel("BAG (years)")
ax.set_title(f"BAG distribution ({best_model})")
ax.legend()
plt.show()

### Corrected BAG by sex

Cole correction is fit on true age only, not sex — a sex gap surviving it is
a genuine model finding, not an artifact of the correction. Mann-Whitney U
(non-parametric, no normality assumption) tests the group difference.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(data=pred, x="sex_label", y="bag_corrected", ax=ax)
sns.stripplot(data=pred, x="sex_label", y="bag_corrected", ax=ax, color="black", alpha=0.15, size=2)
ax.axhline(0, color="k", lw=1, ls=":")
ax.set_xlabel("Sex")
ax.set_ylabel("Corrected BAG (years)")
ax.set_title(f"Corrected BAG by sex ({best_model})")
plt.show()

male_bag = pred.loc[pred["sex_label"] == "Male", "bag_corrected"]
female_bag = pred.loc[pred["sex_label"] == "Female", "bag_corrected"]
u_stat, p_value = stats.mannwhitneyu(male_bag, female_bag)
print(f"Male:   mean BAG={male_bag.mean():+.2f}y, median={male_bag.median():+.2f}y, n={len(male_bag)}")
print(f"Female: mean BAG={female_bag.mean():+.2f}y, median={female_bag.median():+.2f}y, n={len(female_bag)}")
print(f"Mann-Whitney U p-value: {p_value:.4f}")

## 5. MLflow logging

`baseline.run()` is the config-driven wrapper the CLI (`bag models
train-baseline --config ...`) calls — same harness, plus MLflow param/metric
logging to the local SQLite-backed tracking store
(`config/local.yaml`'s `paths.mlflow_dir`). Runs the two checked-in configs
and lists what's logged so far.

In [ ]:
for config_name in ["baseline.yaml", "baseline_ridge.yaml"]:
    result, info = baseline.run(REPO_ROOT / "config" / "models" / config_name)
    print(info["run_name"], {k: round(v, 3) for k, v in result.metrics.items()})

In [ ]:
import mlflow

mlflow.set_tracking_uri(f"sqlite:///{get_path('mlflow_dir') / 'mlflow.db'}")
runs = mlflow.search_runs(experiment_names=["bagpipe-baseline"])
runs[["tags.mlflow.runName", "metrics.mae_raw", "metrics.mae_corrected", "metrics.r2_corrected"]]

## Next steps

Per `docs/DESIGN.md` §7 Phase 2: port the per-region stacked ensemble as
another registered model on these same splits, fine-tune SFCN once the GPU
driver is installed, and promote the best baseline to `models_registry` as
the v1 production model. Run `mlflow ui --backend-store-uri sqlite:///<mlflow_dir>/mlflow.db`
to browse all logged runs interactively.